In [6]:
# =========================================
# BASIC LIBRARIES
# =========================================

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import re
import nltk
import spacy
from wordcloud import WordCloud

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    LSTM,
    Bidirectional,
    Embedding,
    Dropout,
    GlobalMaxPooling1D,
    TextVectorization
)

# Explainability
import shap

# Transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
warnings.filterwarnings('ignore')



In [11]:
# ============================================================
# LOAD DATASETS
# ============================================================

accident_df = pd.read_csv('../data/processed_data.csv')
fire_df = pd.read_csv('../data/industrial_fire_risk_data.csv')

print(accident_df.shape)
print(fire_df.shape)

(425, 16)
(100000, 20)


In [12]:
# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

accident_df.columns = accident_df.columns.str.lower().str.replace(' ', '_')
fire_df.columns = fire_df.columns.str.lower().str.replace(' ', '_')

print(accident_df.columns)
print(fire_df.columns)

Index(['date', 'country', 'local', 'industry_sector', 'accident_level',
       'potential_accident_level', 'gender', 'employee_type', 'critical_risk',
       'description', 'year', 'month', 'day', 'weekday', 'weekofyear',
       'season'],
      dtype='str')
Index(['date', 'time', 'factory', 'region', 'shift', 'workers', 'exp',
       'training', 'temp', 'pressure', 'humidity', 'vibration', 'speed', 'age',
       'service_days', 'gas', 'sparks', 'alarm', 'risk', 'accident'],
      dtype='str')


In [13]:
# ============================================================
# CREATE SEVERITY SCORE
# ============================================================

severity_map = {
    'I':1,
    'II':2,
    'III':3,
    'IV':4,
    'V':5,
    'VI':6
}

accident_df['severity_score'] = accident_df['accident_level'].map(severity_map)

accident_df['high_severity'] = np.where(
    accident_df['severity_score'] >= 4,
    1,
    0
)

In [14]:
# ============================================================
# MERGE DATASETS
# ============================================================

fire_sample = fire_df.sample(
    n=len(accident_df),
    random_state=42,
    replace=True
).reset_index(drop=True)

accident_df = accident_df.reset_index(drop=True)

merged_df = pd.concat([
    accident_df,
    fire_sample
], axis=1)

print(merged_df.shape)

(425, 38)


In [22]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    text = re.sub(r'\\s+', ' ', text).strip()

    return text

merged_df['clean_description'] = merged_df['description'].apply(clean_text)

In [23]:
# ============================================================
# ROOT CAUSE EXTRACTION
# ============================================================

root_cause_keywords = {

    'equipment_failure': [
        'machine',
        'equipment',
        'motor',
        'pump',
        'valve'
    ],

    'human_error': [
        'operator',
        'worker',
        'manual',
        'mistake'
    ],

    'fire_explosion': [
        'fire',
        'explosion',
        'gas',
        'smoke'
    ],

    'electrical': [
        'electrical',
        'shock',
        'voltage'
    ],

    'fall_slip': [
        'fall',
        'trip',
        'slip'
    ]
}


def identify_root_cause(text):

    text = text.lower()

    causes = []

    for category, words in root_cause_keywords.items():

        for word in words:

            if word in text:
                causes.append(category)
                break

    if len(causes) == 0:
        return 'other'

    return ', '.join(causes)


merged_df['root_causes'] = merged_df['clean_description'].apply(
    identify_root_cause
)

In [25]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

merged_df['description_length'] = merged_df['clean_description'].apply(len)

merged_df['word_count'] = merged_df['clean_description'].apply(
    lambda x: len(x.split())
)

In [26]:
merged_df.to_csv('../data/MasterABT.csv', index=False)